# 02 — Симулятор и простые стратегии

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from simulator import *
from agents import *
from experiments import *
assert Path('data/processed.npz').exists(), 'Сначала выполните data_pipeline.py'
data=load_data()
plt.rcParams.update({'figure.dpi':110,'axes.grid':True,'grid.alpha':.2})


Маршрутизация всех запросов — JCH по ключу. Допустимо от 2 до 8 шардов, действие −1/0/+1 на границе 15-минутного интервала. Решение принимается до запросов текущего интервала. Нагрузка — число запросов, одинаковая вычислительная цена каждого запроса — допущение.

Для переноса поддерживаем живые объекты, созданные наблюдаемыми set, с последним размером key+value и TTL. Повторная запись заменяет объект, истечение TTL удаляет его. get не создаёт новую запись. Начальное состояние пустое: отсутствующие ранее созданные объекты не восстанавливаются по будущим данным. Это модель write-defined inventory, не точное восстановление настоящего кеша: нет результатов успешности set и вытеснения.

Стоимость = n + 20×excess/capacity + 4×moved_bytes/train_median_live_bytes. Перенос мгновенный, оплачивается по живым объектам до текущего интервала. Повторный перенос оплачивается снова. Нет реплик, реальных задержек, измерения CPU, сетевой конкуренции или очереди. Превышение мощности не является потерянными запросами или latency.

In [2]:
import unittest
result=unittest.TextTestRunner(verbosity=2).run(unittest.defaultTestLoader.discover('tests'))
assert result.wasSuccessful()
if not Path('results/config.json').exists(): initialize()
c=config()
display(pd.Series(asdict(c)))

test_accumulated_repeated_move_cost (test_project.Tests) ... 

ok
test_causal_observation (test_project.Tests) ... 

ok
test_env_bounds (test_project.Tests) ... 

ok
test_gradient_and_optimizer (test_project.Tests) ... 

ok
test_inventory_overwrite_expiry_delete (test_project.Tests) ... 

...

ok
test_train_only_calibration (test_project.Tests) ... 

ok
test_training_ignores_holdout (test_project.Tests) ... 

ok
test_uint64 (test_project.Tests) ... 

ok

----------------------------------------------------------------------
Ran 10 tests in 0.038s

OK


capacity               5409.0
migration_scale    20686465.5
shard_price               1.0
overload_price           20.0
migration_price           4.0
dtype: float64

Фиксированные baselines: 2–8 шардов, бесплатное первоначальное размещение. Пороговая стратегия начинает с двух; расширяется при превышении верхнего порога прошлой максимальной загрузки и сокращается ниже нижнего. Шесть пар порогов выбираются на validation. Лучшее фиксированное число шардов также выбирается на validation, а не на test.

Validation/test — отдельные эпизоды. До начала периода историческая живая база размещена по начальной схеме; её дальнейшая миграция платная. Предыдущие 4 интервала нагрузки доступны при старте. Во всех динамических стратегиях начальное число шардов — 2.

In [3]:
display(run_baselines())
display(pd.read_csv('results/baseline_train.csv',index_col=0))
print('Выбор по validation:',Path('results/baseline_selection.json').read_text())

Baselines validated: fixed_6; threshold (1.0, 0.4)


,up,down,cost,server_cost,overload_cost,migration_cost,overloaded_slots,excess_requests,moved_mb,mean_shards,changes
0,0.8,0.4,1220.970751,753.0,462.935848,5.034903,67,125201,26.038588,7.843750,6
1,0.8,0.6,1220.970751,753.0,462.935848,5.034903,67,125201,26.038588,7.843750,6
2,1.0,0.4,1181.636429,713.0,463.667961,4.968468,68,125399,25.695012,7.427083,6
3,1.0,0.6,1181.636429,713.0,463.667961,4.968468,68,125399,25.695012,7.427083,6
4,1.2,0.4,1202.765501,689.0,508.851913,4.913588,87,137619,25.411192,7.177083,6
5,1.2,0.6,1202.765501,689.0,508.851913,4.913588,87,137619,25.411192,7.177083,6


,cost,server_cost,overload_cost,migration_cost,overloaded_slots,excess_requests,moved_mb,mean_shards,changes
fixed_2,4934.973563,768.0,4166.973563,0.000000,368.0,1126958.0,0.00000,2.000000,0.0
fixed_3,2424.427436,1152.0,1272.427436,0.000000,198.0,344128.0,0.00000,3.000000,0.0
fixed_4,2139.523757,1536.0,603.523757,0.000000,96.0,163223.0,0.00000,4.000000,0.0
fixed_5,2370.390091,1920.0,450.390091,0.000000,80.0,121808.0,0.00000,5.000000,0.0
fixed_6,2334.545387,2304.0,30.545387,0.000000,40.0,8261.0,0.00000,6.000000,0.0
fixed_7,2688.732113,2688.0,0.732113,0.000000,4.0,198.0,0.00000,7.000000,0.0
fixed_8,3072.000000,3072.0,0.000000,0.000000,0.0,0.0,0.00000,8.000000,0.0
threshold,1905.012518,1893.0,8.300980,3.711538,6.0,2245.0,19.19465,4.929688,6.0


Выбор по validation: {
  "up": 1.0,
  "down": 0.4,
  "fixed": "fixed_6"
}
